# 03 Kaggle Train All Ratios (Per Category)

Runs 3 supervised jobs sequentially for one category: ratios `[100, 50, 10]`.

Day-1 scaling pattern: duplicate this notebook into 6 Kaggle notebooks (one per category) and run them in parallel.

In [ ]:
# Parameters
GITHUB_REPO = "https://github.com/ManuelGamal/Self-Supervised-Industrial-Defect-Detection-System"
GITHUB_BRANCH = "main"

WANDB_ENTITY = "manuelaziz27-ain-shams-university"
WANDB_PROJECT = "defect-detection-supervised"

CATEGORY = "bottle"  # change per notebook: bottle, capsule, carpet, hazelnut, leather, pill
RATIOS = [100, 50, 10]
EPOCHS = 50
BATCH_SIZE = 32

# Cross-validation controls
CV_ENABLED = False
CV_FOLDS = 3
CV_RANDOM_STATE = 42

# Dummy placeholders - replace with your actual paths/slugs
MVTEC_INPUT_DIR = "/kaggle/input/YOUR_MVTEC_DATASET/mvtec-ad"
CKPT_DATASET_SLUG = "YOUR_USERNAME/YOUR_CHECKPOINT_DATASET"

WORKDIR = "/kaggle/working/ssidds"
UPLOAD_DIR = "/kaggle/working/checkpoint_upload_all"
CV_SPLITS_DIR = "/kaggle/working/cv_splits"


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    return client.get_secret(name)

os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY")
os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")

if Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)

run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, WORKDIR])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORKDIR)
run(["wandb", "login", os.environ["WANDB_API_KEY"]])

assert Path(MVTEC_INPUT_DIR).exists(), f"MVTec path not found: {MVTEC_INPUT_DIR}"

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedKFold

def make_cv_split_csv(base_csv: Path, fold_id: int, n_splits: int, random_state: int, out_dir: Path) -> Path:
    df = pd.read_csv(base_csv)
    pool = df[df['split'] != 'test'].copy()  # use train+val as CV pool
    test = df[df['split'] == 'test'].copy()

    if len(pool) == 0:
        raise ValueError(f'No train/val rows found in {base_csv}')

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    indices = list(splitter.split(pool, pool['label']))
    train_idx, val_idx = indices[fold_id]

    fold_train = pool.iloc[train_idx].copy()
    fold_val = pool.iloc[val_idx].copy()
    fold_train['split'] = 'train'
    fold_val['split'] = 'val'

    fold_df = pd.concat([fold_train, fold_val, test], ignore_index=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_dir / f"{CATEGORY}_{base_csv.stem}_fold{fold_id + 1}of{n_splits}.csv"
    fold_df.to_csv(out_csv, index=False)
    return out_csv

def train_one_job(ratio: int, fold_id: int | None = None, split_csv: Path | None = None):
    cmd = [
        sys.executable,
        "-m",
        "src.training.train",
        f"data.root={MVTEC_INPUT_DIR}",
        "data.splits_dir=data/splits",
        f"data.category={CATEGORY}",
        f"data.ratio={ratio}",
        f"training.epochs={EPOCHS}",
        f"training.batch_size={BATCH_SIZE}",
        f"wandb.entity={WANDB_ENTITY}",
        f"wandb.project={WANDB_PROJECT}",
    ]

    if split_csv is not None:
        cmd.append(f"data.split_csv={split_csv}")
    if fold_id is not None:
        cmd.append(f"data.fold_id={fold_id}")
        cmd.append("resume.auto=false")
        cmd.append(f"experiment.seed={CV_RANDOM_STATE + fold_id}")
    else:
        cmd.append("resume.auto=true")

    run(cmd, cwd=WORKDIR)

results = []
for ratio in RATIOS:
    print(f"\n===== CATEGORY={CATEGORY}, RATIO={ratio} =====")

    if not CV_ENABLED:
        train_one_job(ratio=ratio)
        summary_path = Path(WORKDIR) / "results" / "training" / CATEGORY / f"ratio{ratio}" / "run_summary.json"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        summary['ratio'] = ratio
        summary['cv_fold'] = None
        results.append(summary)
        continue

    base_csv = Path(WORKDIR) / "data" / "splits" / f"{CATEGORY}_{ratio}pct_seed42.csv"
    for fold_id in range(CV_FOLDS):
        print(f"--- Fold {fold_id + 1}/{CV_FOLDS}")
        fold_csv = make_cv_split_csv(
            base_csv=base_csv,
            fold_id=fold_id,
            n_splits=CV_FOLDS,
            random_state=CV_RANDOM_STATE,
            out_dir=Path(CV_SPLITS_DIR),
        )
        train_one_job(ratio=ratio, fold_id=fold_id + 1, split_csv=fold_csv)

        summary_path = Path(WORKDIR) / "results" / "training" / CATEGORY / f"ratio{ratio}" / f"fold{fold_id + 1}" / "run_summary.json"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        summary['ratio'] = ratio
        summary['cv_fold'] = fold_id + 1
        summary['cv_fold_total'] = CV_FOLDS
        results.append(summary)

pd.DataFrame(results)


In [ ]:
upload_dir = Path(UPLOAD_DIR)
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True, exist_ok=True)

rows = []
for ratio in RATIOS:
    if not CV_ENABLED:
        summary_path = Path(WORKDIR) / "results" / "training" / CATEGORY / f"ratio{ratio}" / "run_summary.json"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        best_model = Path(summary.get("best_model_path", ""))

        if best_model.exists():
            shutil.copy2(best_model, upload_dir / f"{CATEGORY}_r{ratio}_best.ckpt")

        last_ckpt = Path(WORKDIR) / "checkpoints" / "supervised" / CATEGORY / f"ratio{ratio}" / "last.ckpt"
        if last_ckpt.exists():
            shutil.copy2(last_ckpt, upload_dir / f"{CATEGORY}_r{ratio}_last.ckpt")

        shutil.copy2(summary_path, upload_dir / f"{CATEGORY}_r{ratio}_summary.json")
        rows.append({"category": CATEGORY, "ratio": ratio, "fold": None, "best_model_path": summary.get("best_model_path", "")})
        continue

    for fold_id in range(1, CV_FOLDS + 1):
        summary_path = Path(WORKDIR) / "results" / "training" / CATEGORY / f"ratio{ratio}" / f"fold{fold_id}" / "run_summary.json"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        best_model = Path(summary.get("best_model_path", ""))

        if best_model.exists():
            shutil.copy2(best_model, upload_dir / f"{CATEGORY}_r{ratio}_f{fold_id}_best.ckpt")

        last_ckpt = Path(WORKDIR) / "checkpoints" / "supervised" / CATEGORY / f"ratio{ratio}" / f"fold{fold_id}" / "last.ckpt"
        if last_ckpt.exists():
            shutil.copy2(last_ckpt, upload_dir / f"{CATEGORY}_r{ratio}_f{fold_id}_last.ckpt")

        shutil.copy2(summary_path, upload_dir / f"{CATEGORY}_r{ratio}_f{fold_id}_summary.json")
        rows.append({"category": CATEGORY, "ratio": ratio, "fold": fold_id, "best_model_path": summary.get("best_model_path", "")})

pd.DataFrame(rows).to_csv(upload_dir / f"{CATEGORY}_run_index.csv", index=False)

metadata = {
    "title": "Defect Detection Supervised Checkpoints",
    "id": CKPT_DATASET_SLUG,
    "licenses": [{"name": "CC0-1.0"}],
}
(upload_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

suffix = f"with {CV_FOLDS} folds" if CV_ENABLED else "no CV"
version_note = f"{CATEGORY}: ratios {RATIOS} supervised checkpoints ({suffix})"
run(["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", version_note, "-r", "zip"])

print("Uploaded files:")
for p in sorted(upload_dir.glob("*")):
    print(" -", p.name)
